# ScoutTrainer — GPU pipeline on Colab

Runs the heavy perception stage on Colab's free GPU (minutes instead of hours), then hands you results to browse in your **local** dashboard.

**Before running:** `Runtime → Change runtime type → T4 GPU`.

**Nothing to upload** — cell 1 clones the repo. Push your local changes first (`git add -A && git commit -m ... && git push`), otherwise Colab runs the old code.

**YouTube URLs usually fail on Colab** (datacenter IPs get bot-checked). Uploading the video file in step 3A is the reliable route.

**Reference points are optional** — without them the pipeline runs in camera-relative mode and rates players on on-ball actions.

In [ ]:
# 1) Clone the project (re-run any time to pick up pushed changes)
!rm -rf /content/scout
!git clone https://github.com/kpputhiyattil/scout-agent.git /content/scout
%cd /content/scout
!git log --oneline -1
!nvidia-smi -L

In [ ]:
# 2) Install deps (Colab already has torch+CUDA and ffmpeg). Deno = JS runtime for yt-dlp.
!pip -q install -e ".[perception]" 2>&1 | tail -1
!curl -fsSL https://deno.land/install.sh | DENO_INSTALL=/usr/local sh -s -- -y >/dev/null 2>&1
import torch; print('CUDA available:', torch.cuda.is_available())

## 3) Choose ONE source cell — 3A is recommended

In [ ]:
# 3A) RECOMMENDED — upload a video file from your PC (no YouTube blocking)
from google.colab import files
import pathlib, shutil, re
vdir = pathlib.Path('data/videos'); vdir.mkdir(parents=True, exist_ok=True)
vid = files.upload()                                 # 'Choose Files' button appears below
name = next(iter(vid))
safe = re.sub(r'[^A-Za-z0-9._-]+', '_', name)        # spaces break shell arguments
shutil.move(name, vdir / safe)
SOURCE = ['--file', str(vdir / safe)]
print('using', vdir / safe)

In [ ]:
# 3B) YouTube URL — often blocked on Colab, try 3C if it fails
SOURCE = ['--url', 'https://www.youtube.com/watch?v=E9GLiV_jfro']  # <-- change me

In [ ]:
# 3C) YouTube URL + cookies (clears most bot checks)
#     Export cookies.txt from a logged-in browser with a 'Get cookies.txt' extension.
#     Use a throwaway Google account — cookies are login credentials.
import os
from google.colab import files
ck = files.upload()                                  # pick cookies.txt
os.environ['SCOUT_YTDLP_COOKIES'] = '/content/' + next(iter(ck))
SOURCE = ['--url', 'https://www.youtube.com/watch?v=E9GLiV_jfro']  # <-- change me

In [ ]:
# 4) Optional extras — leave as '' to skip
REF_POINTS = ''   # '' = camera-relative mode (any footage); or 'refs.json' from cell 4a
ROSTER = ''       # roster.csv: jersey_number,name

In [ ]:
# 5) Run the pipeline on GPU (yolov8x is auto-selected when CUDA is available)
import os, shlex
os.environ['SCOUT_DEVICE'] = 'cuda'
args = list(SOURCE)
if REF_POINTS: args += ['--ref-points', REF_POINTS]
if ROSTER: args += ['--roster', ROSTER]
cmd = shlex.join(args)   # quotes any path containing spaces
!python -m scout.pipeline {cmd}

In [ ]:
# 6) Download results — extract into your LOCAL project's data/ folder, then run
#       streamlit run app/dashboard.py
#    Do this before the runtime disconnects: /content is wiped on restart.
!cd data && zip -qr /content/scout_results.zip . -x 'videos/*' 'uploads/*'
from google.colab import files
files.download('/content/scout_results.zip')

## 4a) Pitch reference points — OPTIONAL, run only for fixed wide-angle footage

Skip unless the camera is fixed and the whole pitch is visible. Without reference points the pipeline still rates players on on-ball actions (touches, passing, duels, possession won/lost, shots); distance, speed and pitch-position metrics are excluded rather than guessed.

To use: run cell 5 once (it ingests the video), then run the two cells below and re-run cell 5 with `REF_POINTS = 'refs.json'`.

Pitch coordinates: origin `[0, 0]` = left-bottom corner as seen from the camera, pitch 100 m x 64 m — corners are `[0,0]`, `[100,0]`, `[100,64]`, `[0,64]`.

In [ ]:
# 4a-i) Show an ingested frame with pixel axes, to read landmark coordinates off
import cv2, glob, matplotlib.pyplot as plt
vids = sorted(glob.glob('data/matches/*/video.mp4'))
if not vids:
    print('No ingested video yet — run cell 5 first, or skip this section entirely.')
else:
    cap = cv2.VideoCapture(vids[-1])
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(cap.get(cv2.CAP_PROP_FRAME_COUNT) * 0.5))
    ok, frame = cap.read(); cap.release()
    plt.figure(figsize=(16, 9))
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.grid(color='yellow', alpha=0.4)
    plt.xticks(range(0, frame.shape[1], 50), rotation=90, fontsize=6)
    plt.yticks(range(0, frame.shape[0], 50), fontsize=6)
    plt.show()
    print('frame size (w,h):', frame.shape[1], frame.shape[0])

In [ ]:
# 4a-ii) Fill in the pixel coords you read above, then run to write refs.json
import json
refs = {'points': [
    {'px': [112, 596],  'pitch': [0, 0]},      # <-- left-bottom corner
    {'px': [1163, 601], 'pitch': [100, 0]},    # <-- right-bottom corner
    {'px': [986, 82],   'pitch': [100, 64]},   # <-- right-top corner
    {'px': [297, 80],   'pitch': [0, 64]},     # <-- left-top corner
]}
open('refs.json', 'w').write(json.dumps(refs))
REF_POINTS = 'refs.json'
print('wrote refs.json — now re-run cell 5')